# Synaptic puncta detection (pre / post separately)

LoG + annular z-score on channels 0 (pre) and 1 (post) **independently**,
then keep only puncta whose centre lies on (or near) a structural mask
built from soma ∪ dendrite.

Two paths for the structural mask:
- **load** — read pre-computed soma + dendrite `.npy` masks from disk.
- **derive** — compute them on the fly via `run_dendrite_on_image`
  (uses the new `soma_fdt` + `dendrite_frangi` pipelines; default
  overrides reflect the "satisfactory" snapshot tuned in `02c`).

Calibration is on a few random **patches** of one image (no structural
gate, so LoG+z-score is tuned in isolation).  Results are reported on
**5 full images** with the structural gate applied.

Channel layout: `ch0=pre-synaptic, ch1=post-synaptic, ch2=structural`.
107 nm/px at 60x, MIPs in `(C, H, W)` float32 in `[0, 1]`.

## 1. Imports + repo root

In [8]:
%load_ext autoreload
%autoreload 2

import sys
import re
from pathlib import Path

NB_DIR = Path.cwd().resolve()
for p in (NB_DIR, *NB_DIR.parents):
    if (p / "src" / "synaptic_ssl").exists():
        SRC = (p / "src").resolve()
        if str(SRC) not in sys.path:
            sys.path.insert(0, str(SRC))
        break

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skimage.morphology import dilation, disk as morph_disk

from synaptic_ssl.pseudolabels.puncta import (
    PunctaCfg,
    detect_puncta_channel,
    puncta_to_mask,
    filter_by_size_shape,
    restrict_puncta_to_near,
)
from synaptic_ssl.pseudolabels.puncta import resolve_cfg as resolve_puncta_cfg
from synaptic_ssl.pseudolabels.dendrite_frangi import (
    DEFAULT_DENDRITE_CFG, run_dendrite_on_image,
    resolve_cfg as resolve_dendrite_cfg,
)
from synaptic_ssl.pseudolabels.soma_fdt import (
    DEFAULT_SOMA_CFG, run_soma_on_image,
)
from synaptic_ssl.pseudolabels.viz import (
    visualise_structural_overview,
    visualise_puncta_full,
    visualise_puncta_pair,
)
print(f"src: {SRC}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
src: /home/anokhin/anokhver/thesis/src


## 2. Configuration

All knobs in one place. Per-channel LoG + z-score live in `CFG_PRE`
and `CFG_POST` (two `PunctaCfg` instances).

In [9]:
PUNCTA_CONFIG = dict(
    # ---- Data ---------------------------------------------------------
    no_patch_root="/run/media/anokhin/WinDocuments/anokhver/thesis/data/Microscopy_no_patch/20251030",
    pre_channel=0,
    post_channel=1,
    structural_channel=2,
    n_full_images=5,
    file_offset=1,

    # ---- Calibration (patches) ---------------------------------------
    calib_file_index=0,
    calib_n_patches=6,
    calib_patch_size=128,
    calib_seed=0,
    calib_show_struct=True,

    # ---- Structural-mask source --------------------------------------
    struct_source="load",         # "derive" | "load"
    soma_dir="/run/media/anokhin/WinDocuments/anokhver/thesis/data/pseudolabels/soma/20251030",
    dendrite_dir="/run/media/anokhin/WinDocuments/anokhver/thesis/data/pseudolabels/dendrite/20251030",
    save_struct_masks=False,
    near_dilate_px=4,

    # ---- Puncta saving -----------------------------------------------
    pre_dir="/run/media/anokhin/WinDocuments/anokhver/thesis/data/pseudolabels/pre/20251030",
    post_dir="/run/media/anokhin/WinDocuments/anokhver/thesis/data/pseudolabels/post/20251030",
    save_puncta_masks=False,
    save_patch_size=128,

    apply_shape_filter=False,
)

# ---- Per-channel overrides -------------------------------------------
# Edit values here, re-run this cell + the build cell below.
# Only list keys you want to change from DEFAULT_PUNCTA_CFG_PRE / _POST.
PRE_OVERRIDES = dict(
    # log_min_sigma=1.4,
    # log_max_sigma=2.4,
    # log_threshold=0.008,
    # zscore_threshold=3.0,
    # zscore_inner_radius=5,
    # zscore_outer_radius=12,
    # min_size=13,
    # max_size=60,
)

POST_OVERRIDES = dict(
    # log_min_sigma=1.3,
    # log_max_sigma=1.8,
    # log_threshold=0.012,
    zscore_threshold=6.0,
    # zscore_inner_radius=3,
    # zscore_outer_radius=8,
    # min_size=15,
    # max_size=30,
)

# Multiplier for auto-derived absolute-intensity floors (set_bg_floors).
# Higher k -> stricter: rejects faint ghost dots in dark voids.
BG_FLOOR_K_INNER    = 6.0     # min_inner    = bg_med + k * bg_sigma
BG_FLOOR_K_CONTRAST = 4.0     # min_contrast = k * bg_sigma

# ---- Dendrite / soma derivation overrides ----------------------------
DEND_CFG_OVERRIDES = dict(
    fuzzy_sigma=12.0,
    spur_prune_px=40,
    mu_gamma=0.3,
    dend_fdt_cfg=dict(fdt_mode="gaussian", bg_sigma_k=4.0),
    frangi_sigmas=(2.0, 2.5, 3.0, 4.0, 5.0),
    frangi_alpha=0.5,
    hysteresis_low_pct=45.0,
    small_cc_area=300,
    small_cc_proximity_px=30,
    border_rescue_min_area=400,
    border_rescue_min_axis=30,
    border_pad=15,
)
SOMA_CFG_OVERRIDES = dict()


## 3. Derivation: image stats + biology priors

Reads one image, prints background / foreground stats, and reports
the biology -> pixel math behind the per-channel defaults above. Run
once per dataset, then revisit `CFG_PRE` / `CFG_POST` if any stat
looks off.

In [10]:
# ---- Build CFG_PRE / CFG_POST from defaults + overrides ----------------
CFG_PRE  = resolve_puncta_cfg(PRE_OVERRIDES,  channel="pre")
CFG_POST = resolve_puncta_cfg(POST_OVERRIDES, channel="post")


def _channel_stats(img2d, name):
    bg = img2d[img2d <= np.percentile(img2d, 30)]
    med = float(np.median(bg))
    mad = float(np.median(np.abs(bg - med)))
    sigma = 1.4826 * mad
    p99 = float(np.percentile(img2d, 99))
    p999 = float(np.percentile(img2d, 99.9))
    print(f"  {name:>10s}: bg_med={med:.4f}  bg_sigma={sigma:.4f}  "
          f"p99={p99:.3f}  p99.9={p999:.3f}  "
          f"4*sigma_cut={med + 4*sigma:.3f}")
    return dict(bg=med, sigma=sigma, p99=p99, p999=p999)


def _tophat_stats(img2d, cfg, name):
    from skimage.morphology import white_tophat
    r = int(cfg.intensity_tophat_radius)
    flat = white_tophat(img2d, footprint=morph_disk(r)) if r > 0 else img2d
    return _channel_stats(flat, name)


def set_bg_floors(cfg, stats, *, k_inner, k_contrast):
    cfg.zscore_sigma_bg_floor = float(stats["sigma"])
    cfg.zscore_min_inner      = float(stats["bg"] + k_inner * stats["sigma"])
    cfg.zscore_min_contrast   = float(k_contrast * stats["sigma"])
    return cfg


def _summarise(cfg, name):
    return (
        f"{name}: sigma=[{cfg.log_min_sigma}, {cfg.log_max_sigma}] x{cfg.log_num_sigma}  "
        f"log_thr={cfg.log_threshold}  "
        f"tophat_r={cfg.intensity_tophat_radius}  "
        f"z>={cfg.zscore_threshold} (ann {cfg.zscore_inner_radius}/{cfg.zscore_outer_radius}, "
        f"mu_bg={'p'+str(cfg.zscore_bg_percentile) if cfg.zscore_bg_percentile else 'mean'}, "
        f"sg={'MAD' if cfg.zscore_bg_robust_scale else 'std'})  "
        f"abs[in>={cfg.zscore_min_inner:.3f}, dlt>={cfg.zscore_min_contrast:.3f}, "
        f"sg>={cfg.zscore_sigma_bg_floor:.4f}]  "
        f"size=[{cfg.min_size}, {cfg.max_size}]  wh<={cfg.max_wh_ratio}"
    )


# ---- Image stats -------------------------------------------------------
_files = sorted(Path(PUNCTA_CONFIG["no_patch_root"]).glob("*.npy"))
if not _files:
    raise FileNotFoundError(f"no .npy MIPs found under {PUNCTA_CONFIG['no_patch_root']}")
print(f"n_files = {len(_files)}")
print(f"using file [{PUNCTA_CONFIG['calib_file_index']}] = "
      f"{_files[PUNCTA_CONFIG['calib_file_index']].name}")

_img = np.load(_files[PUNCTA_CONFIG["calib_file_index"]]).astype(np.float32)
print(f"shape = {_img.shape}  dtype = {_img.dtype}")
print("\nPer-channel stats:")
for ch, name in [
    (PUNCTA_CONFIG["pre_channel"],         "pre (ch0)"),
    (PUNCTA_CONFIG["post_channel"],        "post (ch1)"),
    (PUNCTA_CONFIG["structural_channel"],  "struct(ch2)"),
]:
    _channel_stats(_img[ch], name)

# ---- Auto-derive absolute-intensity floors from tophatted image --------
print("\nAfter-tophat per-channel stats (used for floor derivation):")
_th_pre  = _tophat_stats(_img[PUNCTA_CONFIG["pre_channel"]],  CFG_PRE,  "pre^TH")
_th_post = _tophat_stats(_img[PUNCTA_CONFIG["post_channel"]], CFG_POST, "post^TH")

set_bg_floors(CFG_PRE,  _th_pre,  k_inner=BG_FLOOR_K_INNER, k_contrast=BG_FLOOR_K_CONTRAST)
set_bg_floors(CFG_POST, _th_post, k_inner=BG_FLOOR_K_INNER, k_contrast=BG_FLOOR_K_CONTRAST)

# ---- Summary -----------------------------------------------------------
print()
print(_summarise(CFG_PRE,  "PRE "))
print(_summarise(CFG_POST, "POST"))
print(f"struct_source = {PUNCTA_CONFIG['struct_source']!r}  "
      f"save = {PUNCTA_CONFIG['save_struct_masks']}  "
      f"near_dilate = {PUNCTA_CONFIG['near_dilate_px']} px")
print(f"calib: file [{PUNCTA_CONFIG['calib_file_index']}], "
      f"{PUNCTA_CONFIG['calib_n_patches']} patches of "
      f"{PUNCTA_CONFIG['calib_patch_size']}^2 px  seed={PUNCTA_CONFIG['calib_seed']}")


n_files = 68
using file [0] = 1_8_KONTROLA_1_Multichannel Z-Stack_20251030_118.npy
shape = (3, 2304, 2304)  dtype = float32

Per-channel stats:
   pre (ch0): bg_med=0.0119  bg_sigma=0.0071  p99=0.506  p99.9=1.000  4*sigma_cut=0.040
  post (ch1): bg_med=0.0185  bg_sigma=0.0055  p99=0.370  p99.9=1.000  4*sigma_cut=0.040
  struct(ch2): bg_med=0.0214  bg_sigma=0.0106  p99=0.382  p99.9=1.000  4*sigma_cut=0.064

After-tophat per-channel stats (used for floor derivation):
      pre^TH: bg_med=0.0095  bg_sigma=0.0071  p99=0.446  p99.9=0.912  4*sigma_cut=0.038
     post^TH: bg_med=0.0111  bg_sigma=0.0055  p99=0.267  p99.9=0.859  4*sigma_cut=0.033

PRE : sigma=[1.4, 2.4] x4  log_thr=0.008  tophat_r=8  z>=2.5 (ann 5/12, mu_bg=p25, sg=MAD)  abs[in>=0.052, dlt>=0.028, sg>=0.0071]  size=[13, 60]  wh<=4.0
POST: sigma=[1.3, 1.8] x3  log_thr=0.012  tophat_r=6  z>=6.0 (ann 3/8, mu_bg=p25, sg=MAD)  abs[in>=0.044, dlt>=0.022, sg>=0.0055]  size=[11, 30]  wh<=3.0
struct_source = 'load'  save = False  near_d

## 4. Helpers

In [11]:
# ---- I/O helpers (patch tiling / stitching) ----------------------------

_PATCH_RE = re.compile(r"_r(\d{2})_c(\d{2})(?:_[a-z]+)?\.npy$")

def _tile_and_save(arr2d: np.ndarray, out_dir: str, stem: str,
                   patch_size: int, dtype=np.uint8) -> int:
    H, W = arr2d.shape
    if H % patch_size or W % patch_size:
        raise ValueError(
            f"image ({H}, {W}) not divisible by save_patch_size={patch_size}; "
            "the saved tiles would not align with the training patches."
        )
    nr, nc = H // patch_size, W // patch_size
    out_dir_p = Path(out_dir); out_dir_p.mkdir(parents=True, exist_ok=True)
    for rr in range(nr):
        for cc in range(nc):
            tile = arr2d[rr*patch_size:(rr+1)*patch_size,
                         cc*patch_size:(cc+1)*patch_size]
            np.save(out_dir_p / f"{stem}_r{rr:02d}_c{cc:02d}.npy",
                    tile.astype(dtype))
    return nr * nc

def _stitch_patches(out_dir: str, stem: str, full_shape, *, suffix: str = "", dtype=bool) -> np.ndarray:
    H, W = full_shape
    pattern = f"{stem}_r??_c??{suffix}.npy"
    files = sorted(Path(out_dir).glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"no patch files matching {pattern} in {out_dir}. "
            "Run with struct_source='derive' + save_struct_masks=True to populate."
        )
    first = np.load(files[0])
    ps = first.shape[0]
    if first.shape != (ps, ps):
        raise ValueError(f"non-square patch {first.shape} for {files[0].name}")
    nr, nc = H // ps, W // ps
    if len(files) != nr * nc:
        raise ValueError(
            f"expected {nr*nc} patches ({nr}x{nc} at ps={ps}) for {stem}, "
            f"got {len(files)} in {out_dir}"
        )
    out = np.zeros((H, W), dtype=dtype)
    for fp in files:
        m = _PATCH_RE.search(fp.name)
        if not m:
            raise ValueError(f"bad patch filename: {fp.name}")
        rr, cc = int(m.group(1)), int(m.group(2))
        out[rr*ps:(rr+1)*ps, cc*ps:(cc+1)*ps] = np.load(fp).astype(dtype)
    return out


# ---- Structural mask builders ------------------------------------------

def build_structural_from_disk(path: Path, cfg: dict, *, full_shape=None) -> dict:
    if full_shape is None:
        full_shape = np.load(path).shape[-2:]
    soma = _stitch_patches(cfg["soma_dir"],     path.stem, full_shape, suffix="_soma", dtype=bool)
    dend = _stitch_patches(cfg["dendrite_dir"], path.stem, full_shape, suffix="_dend", dtype=bool)
    return dict(soma_mask=soma, dendrite_mask=dend, source="load")


def _bubble_dend_cfg(overrides: dict) -> dict:
    out = dict(overrides)
    single = {"fuzzy_sigma": "fuzzy_sigma", "dend_mu_gamma": "mu_gamma"}
    extra = {tgt: out.pop(src) for src, tgt in single.items() if src in out}
    if extra:
        out["dend_fdt_cfg"] = {**(out.get("dend_fdt_cfg") or {}), **extra}
    return out


def build_structural_derive(path: Path, cfg: dict) -> dict:
    bub = _bubble_dend_cfg(DEND_CFG_OVERRIDES)
    soma_extra = dict(SOMA_CFG_OVERRIDES)
    soma_keys = set(DEFAULT_SOMA_CFG.keys())
    soma_cfg = {k: v for k, v in bub.items() if k in soma_keys} | soma_extra
    dend_cfg = {k: v for k, v in bub.items()
                if k in DEFAULT_DENDRITE_CFG or k == "dend_fdt_cfg"}
    dend_fdt_cfg = dend_cfg.pop("dend_fdt_cfg", None)
    _, _, soma_info, dend_info = run_dendrite_on_image(
        path,
        cfg=dend_cfg,
        soma_cfg=soma_cfg,
        dend_fdt_cfg=dend_fdt_cfg,
        structural_channel=cfg["structural_channel"],
    )
    return dict(
        soma_mask=soma_info["mask"].astype(bool),
        dendrite_mask=dend_info["trace"].astype(bool),
        source="derive",
        soma_info=soma_info,
        dend_info=dend_info,
    )


def build_structural(path: Path, cfg: dict, *, full_shape=None) -> dict:
    if cfg["struct_source"] == "load":
        s = build_structural_from_disk(path, cfg, full_shape=full_shape)
    elif cfg["struct_source"] == "derive":
        s = build_structural_derive(path, cfg)
    else:
        raise ValueError(f"struct_source must be 'load' or 'derive', got {cfg['struct_source']!r}")
    structural = s["soma_mask"] | s["dendrite_mask"]
    near = dilation(structural, morph_disk(int(cfg["near_dilate_px"])))
    s["structural_mask"] = structural
    s["near_mask"] = near
    return s


# ---- Mask saving -------------------------------------------------------

def maybe_save_struct(path: Path, struct: dict, cfg: dict) -> int:
    if not (cfg.get("save_struct_masks") and struct.get("source") == "derive"):
        return 0
    ps = int(cfg["save_patch_size"])
    n = _tile_and_save(struct["soma_mask"],     cfg["soma_dir"],     path.stem, ps)
    n += _tile_and_save(struct["dendrite_mask"], cfg["dendrite_dir"], path.stem, ps)
    return n

def maybe_save_puncta(path: Path, pre_mask: np.ndarray,
                      post_mask: np.ndarray, cfg: dict) -> int:
    if not cfg.get("save_puncta_masks"):
        return 0
    ps = int(cfg["save_patch_size"])
    n  = _tile_and_save(pre_mask,  cfg["pre_dir"],  path.stem, ps)
    n += _tile_and_save(post_mask, cfg["post_dir"], path.stem, ps)
    return n


# ---- Calibration utilities ---------------------------------------------

def sample_patches(img: np.ndarray, n: int, size: int, seed: int):
    rng = np.random.default_rng(seed)
    if img.ndim == 2:
        H, W = img.shape
        img3 = img[None]
    elif img.ndim == 3:
        _, H, W = img.shape
        img3 = img
    else:
        raise ValueError(
            f"sample_patches expects a 2D or 3D image; got shape {img.shape}."
        )
    if size > H or size > W:
        raise ValueError(
            f"calib_patch_size={size} is larger than the image "
            f"(H, W)=({H}, {W}). Lower PUNCTA_CONFIG['calib_patch_size']."
        )
    out = []
    for _ in range(n):
        y = int(rng.integers(0, H - size + 1))
        x = int(rng.integers(0, W - size + 1))
        out.append((y, x, img3[:, y:y + size, x:x + size]))
    return out


# ---- Calibration visualisation -----------------------------------------

def _on_near(scored, near_patch):
    """Per-blob bool: rounded centre lies on near_patch (True if no mask)."""
    if near_patch is None:
        return [True] * len(scored)
    H, W = near_patch.shape
    out = []
    for s in scored:
        rr = int(np.clip(round(s["row"]), 0, H - 1))
        cc = int(np.clip(round(s["col"]), 0, W - 1))
        out.append(bool(near_patch[rr, cc]))
    return out

def viz_calibration_patch(patch, raw_pre, scored_pre, raw_post, scored_post,
                          cfg_pre, cfg_post, *, near_patch=None,
                          pre_ch=0, post_ch=1, vmax=0.3, title=""):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    on_pre  = _on_near(scored_pre,  near_patch)
    on_post = _on_near(scored_post, near_patch)
    for row, (name, ch, raw, scored, cfg, color, on_struct) in enumerate([
        ("pre",  pre_ch,   raw_pre,  scored_pre,  cfg_pre,  "lime",    on_pre),
        ("post", post_ch,  raw_post, scored_post, cfg_post, "magenta", on_post),
    ]):
        img = patch[ch]
        axes[row, 0].imshow(img, cmap="gray", vmin=0, vmax=vmax)
        if near_patch is not None:
            rgba = np.zeros((*near_patch.shape, 4))
            rgba[..., 2] = 1.0; rgba[..., 3] = near_patch * 0.18
            axes[row, 0].imshow(rgba)
        axes[row, 0].set_title(
            f"{name} ch{ch}" + ("  (near-mask blue)" if near_patch is not None else "")
        )
        axes[row, 1].imshow(img, cmap="gray", vmin=0, vmax=vmax)
        for r, c, s in raw:
            axes[row, 1].add_patch(Circle(
                (c, r), float(np.sqrt(2) * s),
                fill=False, edgecolor="yellow", linewidth=0.6,
            ))
        axes[row, 1].set_title(f"{name} LoG raw (n={len(raw)})")
        axes[row, 2].imshow(img, cmap="gray", vmin=0, vmax=vmax)
        n_kept = n_kept_on = n_kept_off = 0
        for s, on in zip(scored, on_struct):
            if s["kept"]:
                n_kept += 1
                if on:
                    n_kept_on += 1; col = color
                else:
                    n_kept_off += 1; col = "orange"
            else:
                col = "red"
            axes[row, 2].add_patch(Circle(
                (s["col"], s["row"]), float(np.sqrt(2) * s["sigma"]),
                fill=False, edgecolor=col, linewidth=0.6,
            ))
        if near_patch is not None:
            title2 = (f"{name} kept z>={cfg.zscore_threshold}  "
                      f"on-mask={n_kept_on} off-mask={n_kept_off}  "
                      f"(of {len(scored) or len(raw)})")
        else:
            title2 = (f"{name} kept z>={cfg.zscore_threshold} "
                      f"(n={n_kept}/{len(scored) or len(raw)})")
        axes[row, 2].set_title(title2)
    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    return fig

## 5. Tuning knobs (edit cell 2 above)

All per-channel overrides live in `PRE_OVERRIDES` / `POST_OVERRIDES`
in cell 2. Ghost-rejection strength: `BG_FLOOR_K_INNER` / `BG_FLOOR_K_CONTRAST`.
Re-run cell 2 + cell 3 to rebuild `CFG_PRE` / `CFG_POST`.


In [12]:
# (overrides moved to PUNCTA_CONFIG cell above)
pass

## 6. Calibration -- patches

Sample `calib_n_patches` random patches from `calib_file_index` and
visualise raw vs kept LoG blobs per channel. No structural gate here:
the goal is to tune `log_*` and `zscore_threshold` in isolation.
Random patches may not contain bright structure -- run the cell again
with a different `calib_seed` to resample.

In [13]:
import time

_files = sorted(Path(PUNCTA_CONFIG["no_patch_root"]).glob("*.npy"))
if not _files:
    raise FileNotFoundError(f"no .npy MIPs found under {PUNCTA_CONFIG['no_patch_root']}")
_img = np.load(_files[PUNCTA_CONFIG["calib_file_index"]]).astype(np.float32)
print(f"calibration image: {_files[PUNCTA_CONFIG['calib_file_index']].name}")
print(f"  shape={_img.shape}  dtype={_img.dtype}  "
      f"min={_img.min():.3f}  max={_img.max():.3f}")

_near = None
if PUNCTA_CONFIG["calib_show_struct"]:
    print("  computing structural near-mask once for calibration overlay ...")
    _ts = time.time()
    _struct_calib = build_structural(
        _files[PUNCTA_CONFIG["calib_file_index"]],
        PUNCTA_CONFIG,
        full_shape=_img.shape[1:],
    )
    _near = _struct_calib["near_mask"]
    print(f"  structural [{_struct_calib['source']}]: "
          f"near={_near.mean():.2%}  ({time.time() - _ts:.1f}s)")
_patches = sample_patches(
    _img,
    n=PUNCTA_CONFIG["calib_n_patches"],
    size=PUNCTA_CONFIG["calib_patch_size"],
    seed=PUNCTA_CONFIG["calib_seed"],
)

_all_z_pre, _all_z_post = [], []
_t0 = time.time()
for i, (y, x, patch) in enumerate(_patches):
    raw_pre,  sc_pre,  kept_pre,  _ = detect_puncta_channel(patch[PUNCTA_CONFIG["pre_channel"]],  CFG_PRE,  auto_floors=False)
    raw_post, sc_post, kept_post, _ = detect_puncta_channel(patch[PUNCTA_CONFIG["post_channel"]], CFG_POST, auto_floors=False)
    _all_z_pre  += [s["z"] for s in sc_pre  if not np.isnan(s["z"])]
    _all_z_post += [s["z"] for s in sc_post if not np.isnan(s["z"])]
    print(f"  patch [{i}] @ (y={y}, x={x})  "
          f"pre raw={len(raw_pre):3d} kept={len(kept_pre):3d}  "
          f"post raw={len(raw_post):3d} kept={len(kept_post):3d}")
    _np = _near[y:y+PUNCTA_CONFIG["calib_patch_size"],
                x:x+PUNCTA_CONFIG["calib_patch_size"]] if _near is not None else None
    viz_calibration_patch(
        patch, raw_pre, sc_pre, raw_post, sc_post,
        CFG_PRE, CFG_POST,
        near_patch=_np,
        pre_ch=PUNCTA_CONFIG["pre_channel"],
        post_ch=PUNCTA_CONFIG["post_channel"],
        title=f"patch {i} @ (y={y}, x={x})",
    )
    plt.show()
print(f"calibration done in {time.time() - _t0:.1f}s")

# Z-score histograms across all calibration patches
_, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, vals, name, thr, color in [
    (ax[0], _all_z_pre,  "pre",  CFG_PRE.zscore_threshold,  "C2"),
    (ax[1], _all_z_post, "post", CFG_POST.zscore_threshold, "C3"),
]:
    arr = np.array(vals)
    a.hist(arr, bins=60, color=color, alpha=0.7)
    a.axvline(thr, color="red", ls="--", label=f"thr={thr}")
    a.set_yscale("log")
    a.set_xlabel("annular z-score")
    a.set_title(f"{name}: n={arr.size}  median={np.median(arr):.2f}  "
                f"kept_frac={(arr >= thr).mean():.2%}")
    a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()


calibration image: 1_8_KONTROLA_1_Multichannel Z-Stack_20251030_118.npy
  shape=(3, 2304, 2304)  dtype=float32  min=0.000  max=1.000
  computing structural near-mask once for calibration overlay ...


ValueError: bad patch filename: 1_8_KONTROLA_1_Multichannel Z-Stack_20251030_118_r00_c00_soma.npy

## 7. Full-image run -- 5 images with structural gate

Runs **on the full image** (2304x2304, no patching). Per file: build
the structural mask (load or derive), detect pre + post puncta on the
full image, restrict to centres on `near_mask`, then visualise the
whole frame.

Set `struct_source = "load"` in cell 4 to read previously saved
masks; set to `"derive"` to compute them now (with optional
`save_struct_masks=True` to persist them for later `load` runs).

Each iteration shows a 1x4 figure on the full image (structural +
mask / pre kept / post kept / composite). Approx 20-30 s per image
with `derive`; `load` is much faster.


In [ ]:
import time

_files = sorted(Path(PUNCTA_CONFIG["no_patch_root"]).glob("*.npy"))
if not _files:
    raise FileNotFoundError(f"no .npy MIPs found under {PUNCTA_CONFIG['no_patch_root']}")
_batch = _files[ 4 + PUNCTA_CONFIG["file_offset"]:
                PUNCTA_CONFIG["file_offset"] + PUNCTA_CONFIG["n_full_images"] + 4]
print(f"full-image batch: {len(_batch)} files  source={PUNCTA_CONFIG['struct_source']!r}  "
      f"save={PUNCTA_CONFIG['save_struct_masks']}")

_stats = []
_t0 = time.time()
for i, path in enumerate(_batch):
    _ti = time.time()
    img = np.load(path).astype(np.float32)
    H, W = img.shape[1:]

    struct = build_structural(path, PUNCTA_CONFIG, full_shape=(H, W))

    _, _, kept_pre,  _ = detect_puncta_channel(img[PUNCTA_CONFIG["pre_channel"]],  CFG_PRE,  auto_floors=False)
    _, _, kept_post, _ = detect_puncta_channel(img[PUNCTA_CONFIG["post_channel"]], CFG_POST, auto_floors=False)
    n_pre_raw  = len(kept_pre)
    n_post_raw = len(kept_post)

    kept_pre  = restrict_puncta_to_near(kept_pre,  struct["near_mask"])
    kept_post = restrict_puncta_to_near(kept_post, struct["near_mask"])

    pre_mask  = puncta_to_mask(kept_pre,  (H, W))
    post_mask = puncta_to_mask(kept_post, (H, W))
    if PUNCTA_CONFIG["apply_shape_filter"]:
        pre_mask  = filter_by_size_shape(pre_mask,  CFG_PRE)
        post_mask = filter_by_size_shape(post_mask, CFG_POST)

    n_struct_tiles = maybe_save_struct(path, struct, PUNCTA_CONFIG)
    n_puncta_tiles = maybe_save_puncta(path, pre_mask, post_mask, PUNCTA_CONFIG)

    _vmax = 0.3
    fig, axes = plt.subplots(1, 3, figsize=(36, 12))

    # col 0: structural mask overlay
    visualise_structural_overview(
        img[PUNCTA_CONFIG["structural_channel"]],
        struct["soma_mask"], struct["dendrite_mask"],
        near_mask=struct["near_mask"],
        ax=axes[0], vmax=_vmax,
        title=f"structural [{struct['source']}]  "
              f"soma={struct['soma_mask'].mean():.2%}  "
              f"dend={struct['dendrite_mask'].mean():.2%}  "
              f"near={struct['near_mask'].mean():.2%}",
    )

    # col 1: pre puncta
    visualise_puncta_full(
        img[PUNCTA_CONFIG["pre_channel"]], kept_pre,
        color="lime", ax=axes[1], vmax=_vmax,
        title=f"PRE  n={len(kept_pre)}",
    )

    # col 2: post puncta
    visualise_puncta_full(
        img[PUNCTA_CONFIG["post_channel"]], kept_post,
        color="magenta", ax=axes[2], vmax=_vmax,
        title=f"POST  n={len(kept_post)}",
    )

    fig.suptitle(f"[{i}] {path.stem[:60]}", fontsize=13)
    plt.tight_layout()
    plt.show()

    _stats.append(dict(
        idx=i, name=path.name,
        n_pre_kept_raw=n_pre_raw,   n_pre_on_struct=len(kept_pre),
        n_post_kept_raw=n_post_raw, n_post_on_struct=len(kept_post),
        frac_near=float(struct["near_mask"].mean()),
        sec=time.time() - _ti,
    ))
    print(f"  [{i}] {path.name[:50]:50s}  "
          f"pre {n_pre_raw}->{len(kept_pre):4d}  "
          f"post {n_post_raw}->{len(kept_post):4d}  "
          f"near={struct['near_mask'].mean():.2%}  "
          f"saved={n_struct_tiles + n_puncta_tiles} tiles  "
          f"{_stats[-1]['sec']:.1f}s")

print(f"\nbatch done in {time.time() - _t0:.1f}s")
print(f"per-image kept on structural: "
      f"pre median={np.median([s['n_pre_on_struct'] for s in _stats]):.0f}  "
      f"post median={np.median([s['n_post_on_struct'] for s in _stats]):.0f}  "
      f"near-mask fraction median={np.median([s['frac_near'] for s in _stats]):.2%}")


## References

- Lindeberg, IJCV 1998 — scale-normalised LoG.
- Wang et al., Bioinformatics 2020 — SynQuant local-SNR test.
- Frangi et al., MICCAI 1998 — vesselness (Path B dendrite).
- Saha et al., CVIU 2002 86(3):171-190 — fuzzy distance transform (Path B soma).
- Channel layout for this dataset: ch0 = pre-synaptic, ch1 = post-synaptic, ch2 = structural.